# Coleta do YouTube: caso Jensen Huang × NVIDIA
Período: 18/10/2026 a 18/11/2026. Vídeos em inglês. Uma linha por vídeo e uma tabela agregada por dia.

In [1]:
import os, json, time, pathlib, requests, pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
CHAVES = [k.strip() for k in os.getenv("YOUTUBE_API_KEYS", "").split(",") if k.strip()]
if not CHAVES:
    raise RuntimeError("Defina YOUTUBE_API_KEYS no arquivo .env")
indice = 0

INICIO, FIM = "2026-10-18", "2026-11-18"
FUSO = "America/New_York"
CONSULTA = 'Jensen Huang | "Jensen Huang" NVIDIA'
PASSO, MAX_PAGINAS = 3, 3
URL = "https://www.googleapis.com/youtube/v3"

pathlib.Path("../data/raw").mkdir(parents=True, exist_ok=True)
pathlib.Path("../data/processed").mkdir(parents=True, exist_ok=True)
cache_arq = pathlib.Path("../data/raw/cache_ids.json")
cache = json.loads(cache_arq.read_text()) if cache_arq.exists() else {}
print(len(CHAVES), "chave(s) carregada(s)")

2 chave(s) carregada(s)


## Funções da API
`chamar` troca para a próxima chave quando a cota da atual acaba. `buscar_ids` faz a busca (100 unidades de cota por página) pedindo vídeos em inglês.

In [2]:
def chamar(endpoint, params):
    global indice
    while True:
        r = requests.get(f"{URL}/{endpoint}", params={**params, "key": CHAVES[indice]}).json()
        if "error" not in r:
            return r
        motivo = r["error"]["errors"][0]["reason"]
        if motivo == "quotaExceeded" and indice < len(CHAVES) - 1:
            indice += 1
            print(f"Cota esgotada, trocando para a chave {indice + 1}")
            continue
        raise RuntimeError(f"{motivo}: {r['error']['message']}")


def buscar_ids(ini, fim):
    ids, token = [], None
    for _ in range(MAX_PAGINAS):
        r = chamar("search", {
            "q": CONSULTA, "type": "video", "part": "id", "maxResults": 50,
            "relevanceLanguage": "en",
            "publishedAfter": ini, "publishedBefore": fim, "pageToken": token})
        ids += [i["id"]["videoId"] for i in r.get("items", [])]
        token = r.get("nextPageToken")
        if not token:
            break
    return ids

## Busca dos IDs (gasta cota)
Janelas de 3 dias, com 1 dia de margem nas pontas. O cache guarda cada janela já buscada: se der erro no meio ou você rodar de novo, ele não paga duas vezes. Apague `data/raw/cache_ids.json` (na raiz do projeto) só se quiser refazer a busca do zero.

In [3]:
inicios = pd.date_range(pd.Timestamp(INICIO) - pd.Timedelta(days=1),
                        pd.Timestamp(FIM) + pd.Timedelta(days=1), freq=f"{PASSO}D")
ids = set()
for d in inicios:
    ini = d.strftime("%Y-%m-%dT00:00:00Z")
    fim = (d + pd.Timedelta(days=PASSO)).strftime("%Y-%m-%dT00:00:00Z")
    if ini not in cache:
        cache[ini] = buscar_ids(ini, fim)
        cache_arq.write_text(json.dumps(cache))
        time.sleep(0.2)
    ids.update(cache[ini])
ids = list(ids)
print(len(ids), "IDs encontrados")

1668 IDs encontrados


## Detalhes dos vídeos
Título, canal, data de publicação, views, curtidas e comentários. Custa 1 unidade de cota a cada 50 vídeos.

In [4]:
linhas = []
for i in range(0, len(ids), 50):
    r = chamar("videos", {"part": "snippet,statistics", "id": ",".join(ids[i:i + 50])})
    for v in r.get("items", []):
        s = v.get("statistics", {})
        linhas.append({
            "video_id": v["id"],
            "data_publicacao": v["snippet"]["publishedAt"],
            "titulo": v["snippet"]["title"],
            "canal": v["snippet"]["channelTitle"],
            "views": s.get("viewCount"),
            "curtidas": s.get("likeCount"),
            "comentarios": s.get("commentCount"),
        })
bruto = pd.DataFrame(linhas)
print(len(bruto), "vídeos brutos")

1668 vídeos brutos


## Limpeza, conversão de fuso e taxa de engajamento por vídeo
1. Remove duplicados e converte os números (curtidas ocultas e comentários desativados ficam NaN e valem 0 só no cálculo da taxa).
2. **Conversão de fuso:** `hora_utc` é a hora original da API, `hora_nova_york` é a mesma hora no fuso da bolsa, e `data` é o dia de `hora_nova_york`, usado para cruzar com as ações.
3. Corta o período de 18/10 a 18/11 usando essa `data` e calcula a taxa de engajamento.

In [12]:
df = bruto.drop_duplicates("video_id").copy()

for c in ["views", "curtidas", "comentarios"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["hora_utc"] = pd.to_datetime(df["data_publicacao"], utc=True)
df["hora_nova_york"] = df["hora_utc"].dt.tz_convert(FUSO)
df["data"] = df["hora_nova_york"].dt.date

df = df[(df["data"] >= pd.to_datetime(INICIO).date()) &
        (df["data"] <= pd.to_datetime(FIM).date())].copy()

df["taxa_engajamento"] = (df["curtidas"].fillna(0) + df["comentarios"].fillna(0)) \
    / df["views"].where(df["views"] > 0)
df["data_coleta"] = pd.Timestamp.now(tz="UTC").date()

df = df[(df["data"] >= pd.to_datetime(INICIO).date()) &
        (df["data"] <= pd.to_datetime(FIM).date())].copy()

total_antes = len(df)
df = df[df["titulo"].str.contains("jensen|huang|nvidia", case=False, na=False)].copy()
print(total_antes - len(df), "vídeos removidos por título fora do tema")

df = bruto.drop_duplicates("video_id").copy()

for c in ["views", "curtidas", "comentarios"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["hora_utc"] = pd.to_datetime(df["data_publicacao"], utc=True)
df["hora_nova_york"] = df["hora_utc"].dt.tz_convert(FUSO)
df["data"] = df["hora_nova_york"].dt.date

df = df[(df["data"] >= pd.to_datetime(INICIO).date()) &
        (df["data"] <= pd.to_datetime(FIM).date())].copy()

total_antes = len(df)
df = df[df["titulo"].str.contains("jensen|huang|nvidia", case=False, na=False)].copy()

df["taxa_engajamento"] = (df["curtidas"].fillna(0) + df["comentarios"].fillna(0)) \
    / df["views"].where(df["views"] > 0)
df["data_coleta"] = pd.Timestamp.now(tz="UTC").date()

df = df.sort_values("hora_utc").reset_index(drop=True)
df.to_csv("../data/raw/youtube_videos.csv", index=False)
print(len(bruto) - len(bruto.drop_duplicates("video_id")), "duplicados removidos |", len(df), "vídeos no período")

50 vídeos removidos por título fora do tema
0 duplicados removidos | 1504 vídeos no período


## Agregação por dia
A taxa diária é ponderada: soma(curtidas + comentários) / soma(views). A coluna `dia_util` marca segunda a sexta, para o join com a bolsa.

In [6]:
diario = df.groupby("data").agg(
    n_videos=("video_id", "count"), views=("views", "sum"),
    curtidas=("curtidas", "sum"), comentarios=("comentarios", "sum")).reset_index()
diario["taxa_engajamento"] = (diario["curtidas"] + diario["comentarios"]) / diario["views"]
diario["dia_util"] = pd.to_datetime(diario["data"]).dt.dayofweek < 5
diario.to_csv("../data/processed/youtube_diario.csv", index=False)
print(len(diario), "dias |", int(diario["dia_util"].sum()), "dias úteis")

32 dias | 23 dias úteis


C:\Users\eaime\AppData\Local\Temp\ipykernel_16752\1944933690.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  diario["dia_util"] = pd.to_datetime(diario["data"]).dt.dayofweek < 5


## Conferência
Confira o menor e o maior dia, os valores ausentes e uma amostra de títulos (para ver se há ruído fora do tema ou fora do inglês).

In [7]:
print(df["data"].min(), "->", df["data"].max())
print(df[["views", "curtidas", "comentarios"]].isna().sum())
df[["hora_utc", "hora_nova_york", "data", "titulo", "canal", "views", "curtidas", "comentarios", "taxa_engajamento"]].head(10)

2026-10-18 -> 2026-11-18
views           0
curtidas       31
comentarios    21
dtype: int64


,hora_utc,hora_nova_york,data,titulo,canal,views,curtidas,comentarios,taxa_engajamento
0,2026-10-18 04:00:19+00:00,2026-10-18 00:00:19-04:00,2026-10-18,Jensen Huang on China's AI labs | Lex Fridman ...,Lex Clips,27510,230.0,26.0,0.009306
1,2026-10-18 04:00:20+00:00,2026-10-18 00:00:20-04:00,2026-10-18,This Week in AI | Anthropic's Clark warns IMF ...,SanMitra Tech Solutions,682,2.0,0.0,0.002933
2,2026-10-18 04:30:03+00:00,2026-10-18 00:30:03-04:00,2026-10-18,Jensen Huang's parents regret his gift #shorts,The Build Story,1477,8.0,0.0,0.005416
3,2026-10-18 07:47:04+00:00,2026-10-18 03:47:04-04:00,2026-10-18,Jensen Huang: NVIDIA without the AI labs is no...,Clipvinci,261,2.0,0.0,0.007663
4,2026-10-18 08:08:24+00:00,2026-10-18 04:08:24-04:00,2026-10-18,Jansen Huang on the one thing success requires,TechPulse,1496,23.0,0.0,0.015374
5,2026-10-18 08:58:32+00:00,2026-10-18 04:58:32-04:00,2026-10-18,Jensen Huang Realized NVIDIA's Name Has a Conn...,Nuvora,4848,46.0,0.0,0.009488
6,2026-10-18 09:15:00+00:00,2026-10-18 05:15:00-04:00,2026-10-18,😬NVIDIA Just Made Your Personal Computer Obsolete,Oladimeji Ipaye,1256,3.0,4.0,0.005573
7,2026-10-18 10:23:00+00:00,2026-10-18 06:23:00-04:00,2026-10-18,NVIDIA wants to cool the world’s hottest compu...,Statecraft,1403,16.0,0.0,0.011404
8,2026-10-18 11:27:31+00:00,2026-10-18 07:27:31-04:00,2026-10-18,Huang: Denny's best dishwasher #shorts,Business News Today,5052,93.0,2.0,0.018804
9,2026-10-18 11:39:34+00:00,2026-10-18 07:39:34-04:00,2026-10-18,Jensen Huang'ın Volvo Kararı #Shorts,Yapay Zeka Avcısı,5,0.0,0.0,0.000000


In [8]:
diario

,data,n_videos,views,curtidas,comentarios,taxa_engajamento,dia_util
0,2026-10-18,35,199715,3595.0,442.0,0.020214,False
1,2026-10-19,47,810514,6171.0,731.0,0.008516,True
2,2026-10-20,29,237578,3035.0,313.0,0.014092,True
3,2026-10-21,45,4211163,26762.0,3299.0,0.007138,True
4,2026-10-22,72,4357277,42006.0,2100.0,0.010122,True
5,2026-10-23,50,32667518,210058.0,2540.0,0.006508,True
6,2026-10-24,45,607117,9776.0,691.0,0.017240,False
7,2026-10-25,40,4502897,75903.0,3484.0,0.017630,False
8,2026-10-26,50,6184252,105350.0,1321.0,0.017249,True
9,2026-10-27,48,360720,6182.0,249.0,0.017828,True
